In [26]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')


In [27]:
def load_data(p='ml-100k/'):
    d = pd.read_csv(p + 'u.data', sep='\t', names=['u', 'i', 'r', 't'])
    info = pd.read_csv(p + 'u.user', sep='|', names=['u', 'age', 'g', 'occ', 'z'])
    u_list = sorted(d['u'].unique())
    u2i = {uid: i for i, uid in enumerate(u_list)}
    y = info.sort_values('u')['occ'].astype('category').cat.codes.values
    return d, u2i, y

In [28]:
def build_h(df, u2i):
    n = len(u2i)
    edges = []
    s = []
    for mid, g in df.groupby('i'):
        p_nodes = [u2i[uid] for uid in g[g['r'] >= 4]['u']]
        if len(p_nodes) >= 2:
            edges.append(p_nodes)
            s.append(1)
        n_nodes = [u2i[uid] for uid in g[g['r'] <= 2]['u']]
        if len(n_nodes) >= 2:
            edges.append(n_nodes)
            s.append(-1)
    h = np.zeros((n, len(edges)))
    for j, nodes in enumerate(edges):
        h[nodes, j] = 1
    return h, np.array(s)

In [29]:
def run_ashd(h, s_init, k=10, iters=50, dim=128):
    n, m = h.shape
    eps = 1e-10
    np.random.seed(42)
    x = np.random.normal(0, 0.1, (n, dim))
    x /= (np.linalg.norm(x, axis=1, keepdims=True) + eps)
    w = s_init.astype(float)
    de = np.sum(h, axis=0) + eps
    de_inv = np.diag(1.0 / de)
    
    for t in range(iters):
        dv = np.sum(h @ np.diag(np.abs(w)), axis=1) + eps
        dv_inv_sqrt = np.diag(1.0 / np.sqrt(dv))
        p = dv_inv_sqrt @ h @ np.diag(w) @ de_inv @ h.T @ dv_inv_sqrt
        x = p @ x
        x /= (np.linalg.norm(x, axis=1, keepdims=True) + eps)
        
        if t > 0 and t % 2 == 0:
            for j in range(m):
                idx = np.where(h[:, j] > 0)[0]
                if len(idx) >= 2:
                    sim = cosine_similarity(x[idx])
                    avg = (np.sum(sim) - len(idx)) / (len(idx) * (len(idx) - 1) + eps)
                    w[j] = np.clip(w[j] + 0.1 * avg, -1, 1)
    return x

In [30]:
def get_mod(h, labs):
    n, m = h.shape
    dv = np.sum(h, axis=1)
    de = np.sum(h, axis=0)
    de[de == 0] = 1
    q = 0.0
    for c in np.unique(labs):
        nodes = np.where(labs == c)[0]
        vol = np.sum(dv[nodes])
        for e in range(m):
            overlap = len(np.intersect1d(np.where(h[:, e] > 0)[0], nodes))
            if overlap > 0:
                q += (overlap / de[e]) - (vol / (2 * m))
    return q / m

if __name__ == "__main__":
    df, u_map, y_true = load_data()
    h, s = build_h(df, u_map)
    
    emb = run_ashd(h, s)
    km = KMeans(n_clusters=10, n_init=20, random_state=42)
    y_pred = km.fit_predict(emb)
    
    print("NMI:", normalized_mutual_info_score(y_true, y_pred))
    print("Modularity:", get_mod(h, y_pred))
    print("Silhouette:", silhouette_score(emb, y_pred))

NMI: 0.06286035713734889
Modularity: -8.334844088631
Silhouette: 0.5000948745140784


In [31]:
def load_data(p='ml-100k/'):
    d = pd.read_csv(p + 'u.data',sep='\t',names=['u', 'i', 'r', 't'])
    info = pd.read_csv(p + 'u.user',sep='|',names=['u', 'age', 'g', 'occ', 'z'])
    u_list = sorted(d['u'].unique())
    u2i = {uid: i for i, uid in enumerate(u_list)}
    info = info[info['u'].isin(u_list)].sort_values('u')
    y = info['occ'].astype('category').cat.codes.values
    return d, u2i, y


def build_h(df, u2i):
    n = len(u2i)
    edges = []
    signs = []

    for mid, group in df.groupby('i'):

        positive_nodes = [
            u2i[uid]
            for uid in group[group['r'] >= 4]['u']
            if uid in u2i]

        if len(positive_nodes) >= 2:
            edges.append(positive_nodes)
            signs.append(1.0)

        negative_nodes = [
            u2i[uid]
            for uid in group[group['r'] <= 2]['u']
            if uid in u2i]

        if len(negative_nodes) >= 2:
            edges.append(negative_nodes)
            signs.append(-1.0)

    H = np.zeros((n, len(edges)), dtype=float)

    for j, nodes in enumerate(edges):
        H[nodes, j] = 1.0
    return H, np.array(signs)

In [32]:
def run_ashd(H, signs, iters=50, dim=128, alpha=0.1):
    n, m = H.shape
    eps = 1e-10
    np.random.seed(42)

    X = np.random.normal(
        0,
        0.1,
        (n, dim))

    X /= (np.linalg.norm(X,axis=1,keepdims=True) + eps)

    W = signs.astype(float).copy()

    d_e = np.sum(H, axis=0) + eps
    D_e_inv = np.diag(1.0 / d_e)

    for t in range(iters):
        d_v = H @ np.abs(W) + eps
        D_v_inv_sqrt = np.diag(
            1.0 / np.sqrt(d_v))

        W_diag = np.diag(W)

        P = (
            D_v_inv_sqrt
            @ H
            @ W_diag
            @ D_e_inv
            @ H.T
            @ D_v_inv_sqrt
        )

        X_new = P @ X

        X = X_new / (
            np.linalg.norm(
                X_new,
                axis=1,
                keepdims=True
            ) + eps
        )

        if t > 0 and t % 2 == 0:
            for j in range(m):
                nodes = np.where(H[:, j] > 0)[0]
                if len(nodes) < 2:
                    continue
                similarity = cosine_similarity(X[nodes])

                avg_sim = (
                    np.sum(similarity) - len(nodes)
                ) / (
                    len(nodes) * (len(nodes) - 1)
                    + eps
                )

                if W[j] > 0:
                    W[j] = np.clip(
                        W[j] + alpha * avg_sim,
                        0.1,
                        1.0)

                else:
                    W[j] = np.clip(
                        W[j] - alpha * avg_sim,
                        -1.0,
                        -0.1
                    )
    return X

In [33]:
def compute_modularity(H, labels):
    n, m = H.shape

    degree = np.sum(H, axis=1)
    edge_size = np.sum(H, axis=0)

    edge_size[edge_size == 0] = 1

    modularity = 0.0

    for c in np.unique(labels):

        nodes = np.where(labels == c)[0]
        volume = np.sum(degree[nodes])
        for e in range(m):

            edge_nodes = np.where(H[:, e] > 0)[0]
            overlap = len(
                np.intersect1d(
                    edge_nodes,
                    nodes
                )
            )

            if overlap > 0:
                modularity += (
                    overlap / edge_size[e]
                    - volume / (2 * m)
                )

    return modularity / m

In [34]:
def run_plain_kmeans(H, n_clusters):

    model = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=42
    )

    return model.fit_predict(H)


def run_spectral_clustering(H, n_clusters):

    A = H @ H.T

    model = SpectralClustering(
        n_clusters=n_clusters,
        affinity='precomputed',
        n_init=10,
        random_state=42
    )

    return model.fit_predict(A)


def run_clique_expansion_kmeans(
    H,
    n_clusters,
    embed_dim=128
):

    A = H @ H.T

    pca = PCA(
        n_components=min(
            embed_dim,
            A.shape[1]
        )
    )

    X = pca.fit_transform(A)

    model = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=42
    )

    return model.fit_predict(X), X


def run_comparison(
    H,
    signs,
    labels_true,
    n_clusters=10
):
    results = []

    X_ashd = run_ashd(
        H,
        signs,
        iters=50,
        dim=128
    )

    labels_ashd = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=42
    ).fit_predict(X_ashd)

    results.append({
        'Method': 'ASHD',
        'NMI': normalized_mutual_info_score(
            labels_true,
            labels_ashd
        ),
        'Modularity': compute_modularity(
            H,
            labels_ashd
        ),
        'Silhouette': silhouette_score(
            X_ashd,
            labels_ashd
        )
    })


    labels_km = run_plain_kmeans(
        H,
        n_clusters
    )

    results.append({
        'Method': 'Plain KMeans',
        'NMI': normalized_mutual_info_score(
            labels_true,
            labels_km
        ),
        'Modularity': compute_modularity(
            H,
            labels_km
        ),
        'Silhouette': silhouette_score(
            H,
            labels_km
        )
    })


    labels_sc = run_spectral_clustering(
        H,
        n_clusters
    )

    A = H @ H.T

    results.append({
        'Method': 'Spectral Clustering',
        'NMI': normalized_mutual_info_score(
            labels_true,
            labels_sc
        ),
        'Modularity': compute_modularity(
            H,
            labels_sc
        ),
        'Silhouette': silhouette_score(
            A,
            labels_sc
        )
    })

    labels_ce, X_ce = run_clique_expansion_kmeans(
        H,
        n_clusters,
        embed_dim=128
    )

    results.append({
        'Method': 'Clique Expansion',
        'NMI': normalized_mutual_info_score(
            labels_true,
            labels_ce
        ),
        'Modularity': compute_modularity(
            H,
            labels_ce
        ),
        'Silhouette': silhouette_score(
            X_ce,
            labels_ce
        )
    })

    return pd.DataFrame(results)


df, u_map, y_true = load_data(
    'ml-100k/'
)

H, signs = build_h(
    df,
    u_map
)

print("Number of users:", H.shape[0])
print("Number of hyperedges:", H.shape[1])
print("Positive hyperedges:", np.sum(signs > 0))
print("Negative hyperedges:", np.sum(signs < 0))

df_results = run_comparison(
    H,
    signs,
    y_true,
    n_clusters=10
)
print(df_results.to_string(index=False))

Number of users: 943
Number of hyperedges: 2598
Positive hyperedges: 1283
Negative hyperedges: 1315
             Method      NMI  Modularity  Silhouette
               ASHD 0.044342   -9.944542    0.597996
       Plain KMeans 0.042794   -8.524440   -0.056783
Spectral Clustering 0.073550   -9.471299   -0.108616
   Clique Expansion 0.056074   -7.418963    0.195696
